# UHPC Dataset


## Data preprocessing

### Import the Data

In [46]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [47]:
df = pd.read_excel('../Datasets/UHPC_data_v2.xlsx', sheet_name=1, header=2)

tw_ei_da_co=len(df[df.columns[53]])
print(f'Number of rows in 28-Day = {tw_ei_da_co}')

Number of rows in 28-Day = 2188


### Initial cleaning

In [48]:
# drop the missing rows where the target value is missing
tw_ei_da = df.columns[53]
df_drop=df.dropna(subset=[tw_ei_da]).copy()
print(f'Number of rows in 28-Day = {len(df_drop[df.columns[53]])}')

Number of rows in 28-Day = 2073


In [49]:
# extracting values from the string cells in curing pressure z.B. 1.2 Mpa for 4 and 5 hrs
pressure_col = df_drop.columns[42] 
df_drop[pressure_col] = df_drop[pressure_col].astype(str).str.extract(r'([0-9.]+)').astype(float)

### Defining the dictionaries

In [50]:
ingredients = {
    # w/o types
    "Silica Fume":      {"amount": df_drop.columns[4],  
                         "type": None, "extras": []},
    "Limestone Powder": {"amount": df_drop.columns[7],  
                         "type": None, "extras": []},
    "Quartz Powder":    {"amount": df_drop.columns[8],  
                         "type": None, "extras": []},
    "Glass Powder":     {"amount": df_drop.columns[9],  
                         "type": None, "extras": []},
    "Rice Husk Ash":    {"amount": df_drop.columns[10], 
                         "type": None, "extras": []},
    "Metakaolin":       {"amount": df_drop.columns[11], 
                         "type": None, "extras": []},
    "Nano-CaCO3":       {"amount": df_drop.columns[15], 
                         "type": None, "extras": []},
    "Nano-Al2O3":       {"amount": df_drop.columns[16], 
                         "type": None, "extras": []},
    "Nano-TiO2":        {"amount": df_drop.columns[17], 
                         "type": None, "extras": []},
    "Nano-Silica":      {"amount": df_drop.columns[18], 
                         "type": None, "extras": []},
    "Water":            {"amount": df_drop.columns[36],
                         "type": None, "extras": []},

    # amount and type
    "Fly Ash":          {"amount": df_drop.columns[5],  
                         "type": df_drop.columns[6],  "extras": []},
    "Slag":              {"amount": df_drop.columns[13], 
                         "type": df_drop.columns[14], "extras": []},
    "Superplasticizer": {"amount": df_drop.columns[38], 
                         "type": df_drop.columns[37], "extras": []},
    "Sustainable Filler": {"amount": df_drop.columns[19], 
                           "type": df_drop.columns[20], "extras": []},

    # amount, type and other parameters
    "Sand": {
        "amount": df_drop.columns[21], 
        "type": df_drop.columns[22], 
        "extras": [df_drop.columns[23]]  # max size
    },
    "Fiber": {
        "amount": df_drop.columns[25], 
        "type": df_drop.columns[24], 
        "extras": [df_drop.columns[26], # length
                   df_drop.columns[27], # diameter
                   df_drop.columns[28], # tensile
                   df_drop.columns[29]] # Young's modulus

    },
    "Synergetic Fiber": {
        "amount": df_drop.columns[31], 
        "type": df_drop.columns[30], 
        "extras": [df_drop.columns[32], # length
                   df_drop.columns[33], # diameter
                   df_drop.columns[34], # tensile strength
                   df_drop.columns[35]] # Young's modulus
    }
}

cement_amt_r   = df.columns[1]
cement_type_r = df.columns[2]
cement_grade_r = df.columns[3]

curing_type_r  = df.columns[39]
curing_temp_r  = df.columns[40]
curing_humid_r = df.columns[41]
curing_press_r = pressure_col

### Feature policy 

In [51]:
# 100, 50, 20
feature_policy = 50

# gather all raw features from the dictionary
amount_cols = [data["amount"] for data in ingredients.values() if data["amount"] is not None]
type_cols = [data["type"] for data in ingredients.values() if data["type"] is not None]
extra_cols = []
for data in ingredients.values():
    extra_cols.extend(data["extras"])



# define curing and cement features
other_raw = [
    cement_amt_r,  # Cement Amount
    cement_type_r,  # Cement Type
    cement_grade_r,  # Cement Grade
    curing_type_r, # Curing Type
    curing_temp_r, # Curing Temp
    curing_humid_r, # Curing Humidity
    pressure_col    # Curing Pressure
]

# combine all features  
all_features = amount_cols + type_cols + extra_cols + other_raw
# Ensure no KeyError by checking what is actually in the dataframe
all_features = [f for f in all_features if f in df_drop.columns]

# calculate missing percentages
missing_perc = df_drop[all_features].isnull().sum() / len(df_drop) * 100

# Policy logic
if feature_policy == 100:
    cols_to_drop = []
    
elif feature_policy == 50:
    cols_to_drop = missing_perc[missing_perc > 50].index.tolist()
    
elif feature_policy == 20:
    cols_to_drop = missing_perc[missing_perc > 20].index.tolist()
    
else:
    raise ValueError("Please choose policy from the provided options: 100, 50, 20")

# execute the drop
df_drop = df_drop.drop(columns=cols_to_drop)


print(f"Policy: {feature_policy}")
print(f"{len(all_features)} total raw features.")
print(f"Dropped {len(cols_to_drop)} columns: {cols_to_drop}")

Policy: 50
41 total raw features.
Dropped 13 columns: ['Amount / Quantity of Fiber .1', 'Fly Ash Type ', 'Type of Slag ', 'Type of Filler ', 'Type of Fiber .1', 'Tensile Strength (MPa)', 'Nominal Young’s modulus, Gpa', 'Length (mm).1', 'Diameter (mm).1', 'Tensile Strength (MPa).1', 'Nominal Young’s modulus, Gpa.1', 'Unnamed: 41', 'Unnamed: 42']


### Semantic Recoding

In [52]:
if feature_policy != 100:
    print("Applying semantic recoding...")
    
    cement_type_col = df.columns[2]
    if cement_type_col in df_drop.columns:
        df_drop[cement_type_col] = df_drop[cement_type_col].fillna("unknown_type")

    # Clean ingredients
    for name, data in ingredients.items():
        amount_col = data["amount"]
        type_col = data["type"]
        extras_col = data["extras"]
        
        if amount_col in df_drop.columns:
            
            df_drop[amount_col] = df_drop[amount_col].astype(str).str.extract(r'([0-9.]+)').astype(float)
            
            # amount = 0 or nan
            not_used = (df_drop[amount_col] == 0) | (df_drop[amount_col].isnull())
            
            # if not used types and extra parameters don't apply
            if type_col is not None and type_col in df_drop.columns:
                df_drop.loc[not_used, type_col] = 'not_applicable'
                
            for extra in extras_col:
                if extra in df_drop.columns:
                    # extract the numbers and force the column to be a float
                    df_drop[extra] = df_drop[extra].astype(str).str.extract(r'([0-9.]+)').astype(float)
                    
                    # assign 0.0
                    df_drop.loc[not_used, extra] = 0.0
                
            # amount > 0 but type is missing
            if type_col is not None and type_col in df_drop.columns:
                missing_type = (df_drop[amount_col] > 0) & (df_drop[type_col].isnull())
                df_drop.loc[missing_type, type_col] = "unknown_type"
            

            if type_col is not None and type_col in df_drop.columns:
                # type exists but amount is missing
                missing_amt = (df_drop[type_col].notnull()) & (df_drop[amount_col].isnull())
            else:
                missing_amt = df_drop[amount_col].isnull()

            # the binary flag column 
            flag_col = f"{name}_Amount_is_missing"
            df_drop[flag_col] = 0.0
            df_drop.loc[missing_amt, flag_col] = 1.0
            
            # keep the original column numeric
            df_drop.loc[missing_amt, amount_col] = 0.0

    # Clean the curing regime
    if curing_type_r in df_drop.columns:
    
        # Safely check which numeric columns 
        has_temp = curing_temp_r in df_drop.columns
        has_humid = curing_humid_r in df_drop.columns
        has_press = pressure_col in df_drop.columns
        
        # Build the has_numeric_data mask
        has_numeric_data = pd.Series(False, index=df_drop.index)
        if has_temp:
            has_numeric_data = has_numeric_data | (df_drop[curing_temp_r] >= 0)
        if has_humid:
            has_numeric_data = has_numeric_data | (df_drop[curing_humid_r] >= 0)
        if has_press:
            has_numeric_data = has_numeric_data | (df_drop[pressure_col] >= 0)

        # if t, h, p are not nan but type is blank    
        type_is_null = df_drop[curing_type_r].isnull()
        df_drop.loc[type_is_null & has_numeric_data, curing_type_r] = "unknown_type"

        # everything is nan 
        df_drop[curing_type_r] = df_drop[curing_type_r].fillna("not_applicable")

        # conditional average imputation for missing temperatures
        unique_types = df_drop[curing_type_r].dropna().unique()
        for c_type in unique_types:
            is_this_type = df_drop[curing_type_r] == c_type
            
            if has_temp:
                type_temp_mean = df_drop.loc[is_this_type, curing_temp_r].mean()
                df_drop.loc[is_this_type & df_drop[curing_temp_r].isnull(), curing_temp_r] = type_temp_mean
            if has_humid:
                type_humid_mean = df_drop.loc[is_this_type, curing_humid_r].mean()
                df_drop.loc[is_this_type & df_drop[curing_humid_r].isnull(), curing_humid_r] = type_humid_mean
            if has_press:
                type_press_mean = df_drop.loc[is_this_type, pressure_col].mean()
                df_drop.loc[is_this_type & df_drop[pressure_col].isnull(), pressure_col] = type_press_mean

else:
    print("Skipping semantic recoding (Full Raw)")

Applying semantic recoding...


### Feature Selection and Data Splitting

In [53]:
# Define the base feature groups
mix_features = [data["amount"] for data in ingredients.values() if data["amount"] is not None]
extra_features = []
for data in ingredients.values():
    extra_features.extend(data["extras"])

missing_flags = [f"{col}_is_missing" for col in mix_features] 

cement_features = [cement_amt_r, cement_type_r, cement_grade_r]

curing_features = [
    curing_type_r, 
    curing_temp_r,
    curing_humid_r,
    pressure_col    # Curing Pressure
] 

# Combine everything and intersect
keep_features_raw = cement_features + mix_features + extra_features + missing_flags + curing_features
keep_features = list(df_drop.columns.intersection(keep_features_raw))

# Define X and Y safely
Y = df_drop['28-Day']
X = df_drop[keep_features]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

### Encoding Target K-Fold and OneHot

In [54]:
Y = df_drop['28-Day']
X = df_drop[keep_features]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


# gather all ingredients type except cement_type
one_hot_encode_cols = [bundle["type"] for bundle in ingredients.values() if bundle["type"] is not None and bundle["type"] in X_train.columns]

if curing_type_r in X_train.columns:
    one_hot_encode_cols.append(curing_type_r)

# handle_unknown='ignore' to prevent test-set crashes
one_enc = OneHotEncoder(categories='auto', sparse_output=False, handle_unknown='ignore')

one_enc_array_train = one_enc.fit_transform(X_train[one_hot_encode_cols])
one_enc_array_test = one_enc.transform(X_test[one_hot_encode_cols])

one_enc_headers = one_enc.get_feature_names_out(one_hot_encode_cols)
one_enc_df_train = pd.DataFrame(one_enc_array_train, columns=one_enc_headers, index=X_train.index)
one_enc_df_test = pd.DataFrame(one_enc_array_test, columns=one_enc_headers, index=X_test.index)

# TargetEncoder with cv
# smooth='auto' calculates the best weighted mean balance
target_encode_cols = [
    col for col in [cement_type_r] 
    if col in X_train.columns
]
enc = TargetEncoder(categories='auto', cv=5, smooth='auto', random_state=42)

# fit and transform the columns
enc_array_train = enc.fit_transform(X_train[target_encode_cols], Y_train)
enc_array_test = enc.transform(X_test[target_encode_cols])

# create a clean DataFrame with the new numerical values
enc_headers = [f"{col}_encoded" for col in target_encode_cols]
cem_enc_df_train = pd.DataFrame(enc_array_train, columns=enc_headers, index=X_train.index)
cem_enc_df_test = pd.DataFrame(enc_array_test, columns=enc_headers, index=X_test.index)

# combining the original columns to drop
dropping_columns = target_encode_cols + one_hot_encode_cols

# combine both df
df_after_enc_train = pd.concat([X_train, cem_enc_df_train, one_enc_df_train], axis=1)
df_after_enc_test = pd.concat([X_test, cem_enc_df_test, one_enc_df_test], axis=1)

X_train_final = df_after_enc_train.drop(columns=dropping_columns)
X_test_final = df_after_enc_test.drop(columns=dropping_columns)

### Formatting numeric features and handling coerced NaNs

In [55]:
# all new numeric features to be scaled
new_numerics = [cement_amt_r, cement_grade_r, curing_temp_r, curing_humid_r, pressure_col]

all_numeric_features = new_numerics + mix_features + extra_features

# Safely find which ones survived the policy drop
valid_numeric_features = [col for col in all_numeric_features if col in X_train_final.columns]

# Iterate over the surviving numeric columns
for col in valid_numeric_features:
    X_train_final[col] = pd.to_numeric(X_train_final[col], errors='coerce')
    X_test_final[col] = pd.to_numeric(X_test_final[col], errors='coerce')

# Fill the NaNs
X_train_final[valid_numeric_features] = X_train_final[valid_numeric_features].fillna(0.0)
X_test_final[valid_numeric_features] = X_test_final[valid_numeric_features].fillna(0.0)

### Scaling

In [56]:
scaler = StandardScaler()

# Use valid_numeric_features instead of mix_features
scaled_features_train = scaler.fit_transform(X_train_final[valid_numeric_features])
scaled_features_test = scaler.transform(X_test_final[valid_numeric_features])

X_train_final[valid_numeric_features] = scaled_features_train
X_test_final[valid_numeric_features] = scaled_features_test

## Model training

### XGBoost initialization

In [57]:
from xgboost import XGBRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, AdaBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [58]:
# Force everything to be strict numbers, turn any weird text into NaN, and fill with 0
X_train_final = X_train_final.apply(pd.to_numeric, errors='coerce').fillna(0.0)
X_test_final = X_test_final.apply(pd.to_numeric, errors='coerce').fillna(0.0)

model_xgb = XGBRegressor(random_state = 42)
model_xgb.fit(X_train_final,Y_train)
Y_pred = model_xgb.predict(X_test_final)
mae = mean_absolute_error(Y_test,Y_pred)
rmse =root_mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

metrics_dict_xgb = {'MAE':[mae], 'RMSE':[rmse], 'R2': [r2]}
metrics_xgb = pd.DataFrame(metrics_dict_xgb)
metrics_xgb

,MAE,RMSE,R2
0,9.992935,14.203209,0.850607


In [59]:
model_hist = HistGradientBoostingRegressor(random_state = 42)
model_hist.fit(X_train_final,Y_train)
Y_pred = model_hist.predict(X_test_final)
mae = mean_absolute_error(Y_test,Y_pred)
rmse =root_mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

metrics_dict_hist = {'MAE':[mae], 'RMSE':[rmse], 'R2': [r2]}
metrics_hist = pd.DataFrame(metrics_dict_hist)
metrics_hist

,MAE,RMSE,R2
0,9.989841,13.871643,0.8575


In [ ]:
model_ada = AdaBoostRegressor(random_state = 42)
model_ada.fit(X_train_final,Y_train)
Y_pred = model_ada.predict(X_test_final)
mae = mean_absolute_error(Y_test,Y_pred)
rmse =root_mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

metrics_dict_ada = {'MAE':[mae], 'RMSE':[rmse], 'R2': [r2]}
metrics_ada = pd.DataFrame(metrics_dict_ada)
metrics_ada

,MAE,RMSE,R2
0,16.303511,20.370822,0.692691
